## Omnimind Agent System

In [14]:
import os
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool
from typing import TypedDict, List
import ast
import numpy as np

### LLM

In [7]:
llm = ChatOpenAI(api_key=os.getenv('OPENAI_SECRET_KEY'),
                 model='gpt-4o-mini',
                 temperature=0)

llm

ChatOpenAI(profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001ED0D8896D0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001ED0DDB1F90>, root_client=<openai.OpenAI object at 0x000001ED0D88AA90>, root_async_client=<openai.AsyncOpenAI object at 0x000001ED0DDB1B10>, model_name='gpt-4o-mini', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

### Agent State

In [ ]:
class AgentState(TypedDict):
    query: str
    tasks: list
    results: list
    final_result: str
    
    retries: int
    valid: bool
    
AgentState

__main__.AgentState

### Tools

#### Percentage

In [20]:
@tool
def percentage_calc_tool(value: float, percentage: float) -> str:
    '''Calculate Percentage'''
    
    print('\n[TOOL: Percentage Calculator]')
    try:
        return str((value*percentage)/100)
    except Exception as e:
        return f"Claculation Error {e}"

#### Mean

In [21]:
@tool
def calculate_mean_tool(numbers: str) -> str:
    '''Calculates mean of numbers separated by comma'''

    print('\n[TOOL: Calculate Mean]')
    try:
        nums= np.array(numbers.split(',')).astype(float)
        return nums.mean()
    except Exception as e:
        return f'Mean Error {e}'

#### Code Validator

In [22]:
@tool
def code_validator_tool(code: str) -> str:
    '''Validates Python code using AST parsing'''

    print('\n[TOOL: Code Validator]')
    try:
        ast.parse(code)
        return "Code issyntactically valid"
    except SyntaxError as e:
        return f"Syntax Error: {e}"
    except IndentationError as e:
        return f'Indentation Error: {e}'
    except Exception as e:
        print(f'Code Validation Error: {e}')

#### Keyword Extractor

In [23]:
@tool
def keyword_extractor_tool(text: str) -> str:
    '''Extracts first 5 words from the given text'''
    
    print('\n[TOOL: Keyword Extractor]')
    try:
        return ','.join(text.split(',')[:5])
    except Exception as e:
        return f'Keyword Extraction Error: {e}'

### Agents

#### Math Agent

In [24]:
math_agent = create_agent(
    model=llm,
    tools=[percentage_calc_tool],

    system_prompt='''You are a math expert. You always use the given tool when needed.'''
)

#### Coding Agent

In [25]:
coding_agent = create_agent(
    model=llm,
    tools=[code_validator_tool],
    
    system_prompt='''You are a coding expert.
    Generate correct Python code and always validate it with the given tool'''
)

#### Data Analysis Agent

In [26]:
data_agent = create_agent(
    model=llm,
    tools=[calculate_mean_tool, keyword_extractor_tool],
    
    system_prompt='''You are a data analyst. Use the given tools to calculate the mean and to extract the keywords as needed'''
)

#### Summary Agent

In [27]:
summary_agent = create_agent(
    model=llm,
    tools=[],
    
    system_prompt='''You are a helpful Summarize Assistant.
Summarise the given text simply and precisly without losing the main context and the information given in the text. '''
)

#### General Agent

In [28]:
general_agent = create_agent(
    model=llm,
    tools=[],
    
    system_prompt='''You are a helpful assistant. You answer the general queries very briefly with in 100 words.'''
)

### Nodes

In [29]:
def input_guardrails(query):
    print('\n[INPUT GUARDRAILS] ', query)
        
    # Empty check
    if not query.strip():
        # return {'valid': False}
        return False
    
    # Keyword check
    banned_keywords = ['hack', 'attack', 'illegal', 'ignore instructions', 'no restrictions', 'system prompt']
    if any(word in query.lower() for word in banned_keywords):
        # return {'valid': False}
        return False
        
    # return {'valid': True}
    return True

In [30]:
def output_guardrails(result):
    print('\n[OUTPUT GUARDRAILS NODE]')

    # Empty check
    if not result.strip():
        return False
    
    return True

In [31]:
def decomposition_node(query):
    print('\n[DECOMPOSITION NODE]')
    #query = state['query']
    
    try:
        prompt = f'''You are a query decomposition assistant. You have the following tools:
- math: Solves math expressions
- code: Generates Python code
- data: Data Analysis Assistant
- summary: Summarizes text
- general: Answers general queries

QUERY:
{query}

Analyze if the query requires MULTIPLE tools to answer completely.
- If YES, break it into smaller sub-queries, each mapped to a specific tool.
- If NO, return the original query mapped to the appropriate tool.

Return ONLY a Python list of dictionaries. No explanation, no markdown.
Example (multiple): [{{"task": "What is 5+5?", "type": "math"}}, {{"task": "Write a Python code to add two numbers", "type": "code"}}]
Example (single): [{{"task": "What is the capital of France?", "type": "general"}}]
'''

        result = llm.invoke(prompt).content.strip()
        result = ast.literal_eval(result) # convert to list
        print('[DECOMPOSED TASKS]')
        print(result)
        return {'tasks': result}
    except Exception as e:
        print(f'Decompsition Error: {e}')
        return {'tasks': query}

In [38]:
tasks = decomposition_node('what is 23 * 45? Explain AI in brief and what are the latest trends ')
print(type(tasks.get('tasks')))

tks = tasks.get('tasks')
for task in tks:
    print(task.get('task'))


[DECOMPOSITION NODE]
[DECOMPOSED TASKS]
[{'task': 'What is 23 * 45?', 'type': 'math'}, {'task': 'Explain AI in brief', 'type': 'general'}, {'task': 'What are the latest trends in AI?', 'type': 'general'}]
<class 'list'>
What is 23 * 45?
Explain AI in brief
What are the latest trends in AI?


In [ ]:
def task_executor(state):
    results = []
    for item in state['tasks']:
        task = item.get('task')
        tool = item.get('type')
        
        if (tool == 'code'):
            result = coding_agent.invoke(
                {
                    "messages": [{'role' : 'user' , 'content' : state['query']}]
                }
            )
            answer = result['messages'][-1].content
        

In [136]:
def validate_node(state):
    print('\n[VALIDATE NODE]')

    res = state.get('final_result', '')
    res = res.strip()

    if res and len(res)>0 and 'error' not in res.lower():
        return {'valid' : True}
    return {'valid':False}

In [137]:
MAX_RETRIES=2
def retry_node(state):
    print('\n[RETRY NODE]')

    retries = state['retries']

    if retries >= MAX_RETRIES:
        return {'route' : 'fallback'}

    return {
        'retries':retries + 1,
        'route' : state['route']
    }

In [138]:
def fallback_node(state):
    print('\n[FALLBACK NODE]')
    try:
        result = llm.invoke(state['query']).content
        return {'final_result':result}
    except Exception as e:
        return {'final_result': 'System Failed Completely'}

### Flow

In [ ]:
graph = StateGraph(AgentState)

# graph.add_node('input_guardrails_n', input_guardrails_node)
# graph.add_node('output_guardrails_n', output_guardrails_node)
graph.add_node('decomposition_n', decomposition_node)
graph.add_node('validate_n', validate_node)
graph.add_node('fallback_n', fallback_node)
graph.add_node('retry_n', retry_node)

In [161]:
graph.set_entry_point('decomposition_n')

graph.add_edge('decomposition_n', 'router_n')
graph.add_conditional_edges(
    'router_n',
    lambda s:s['route'],
    {
        'math': 'math_n',
        'coding' : 'coding_n' ,
        'data analysis': 'data_analysis_n',
        'summary' : 'summary_n',
        'general': 'general_n'
    }
)

graph.add_edge('math_n', 'combine_n')
graph.add_edge('coding_n', 'combine_n')
graph.add_edge('data_analysis_n', 'combine_n')
graph.add_edge('summary_n', 'combine_n')
graph.add_edge('general_n', 'combine_n')

graph.add_edge('combine_n', 'validate_n')

graph.add_conditional_edges(
    'validate_n',
    lambda s: 'valid' if s['valid'] else 'retry',
    {
        'valid': END,
        'retry': 'retry_n'
    }
)

graph.add_conditional_edges(
    'retry_n',
    lambda s:s['route'],
    {
        'math': 'math_n',
        'coding' : 'coding_n' ,
        'data analysis': 'data_analysis_n',
        'summary' : 'summary_n',
        'general': 'general_n',
        'fallback': 'fallback_n',
    }
)

graph.add_edge('fallback_n', END)

app = graph.compile()

### Agent Call

In [164]:
def agent_call():
    while True:
        query = input('Ask (type exit to quit): ')
        
        if query.lower() in ['exit', 'quit', 'stop', 'end']:
            print('exiting......')
            break
        
        result = app.invoke(
            {'query': query, 'retries': 0}
        )
            
        print(f"\nFinal Answer: {result}")

In [165]:
agent_call()


[DECOMPOSITION NODE]
[DECOMPOSED TASKS]
1. General: Identify keywords in the sentence "how are you doing buddy?"

[ROUTER NODE]
\Route:  general

[GENERAL NODE]

[COMBINE NODE]
Final Result: 
Keywords in the sentence "how are you doing buddy?" include "how," "you," and "buddy." These words convey the inquiry about someone's well-being and the informal, friendly tone of the conversation.

[VALIDATE NODE]

Final Answer: {'query': '1. General: Identify keywords in the sentence "how are you doing buddy?"', 'route': 'general', 'general_result': 'Keywords in the sentence "how are you doing buddy?" include "how," "you," and "buddy." These words convey the inquiry about someone\'s well-being and the informal, friendly tone of the conversation.', 'final_result': 'Keywords in the sentence "how are you doing buddy?" include "how," "you," and "buddy." These words convey the inquiry about someone\'s well-being and the informal, friendly tone of the conversation.', 'retries': 0, 'valid': True}
exit

In [166]:
agent_call()


[DECOMPOSITION NODE]
[DECOMPOSED TASKS]
["General: Who won the IPL in 2023?"]

[ROUTER NODE]
\Route:  general

[GENERAL NODE]

[COMBINE NODE]
Final Result: 
The Chennai Super Kings (CSK) won the IPL in 2023, defeating the Gujarat Titans in the final.

[VALIDATE NODE]

Final Answer: {'query': '["General: Who won the IPL in 2023?"]', 'route': 'general', 'general_result': 'The Chennai Super Kings (CSK) won the IPL in 2023, defeating the Gujarat Titans in the final.', 'final_result': 'The Chennai Super Kings (CSK) won the IPL in 2023, defeating the Gujarat Titans in the final.', 'retries': 0, 'valid': True}
exiting......


In [168]:
agent_call()


[DECOMPOSITION NODE]
[DECOMPOSED TASKS]
- Summarize: The Chennai Super Kings (CSK) won the IPL in 2023, defeating the Gujarat Titans in the final.

[ROUTER NODE]
\Route:  summary

[SUMMARY NODE]

[COMBINE NODE]
Final Result: 
The Chennai Super Kings (CSK) won the 2023 IPL by defeating the Gujarat Titans in the final.

[VALIDATE NODE]

Final Answer: {'query': '- Summarize: The Chennai Super Kings (CSK) won the IPL in 2023, defeating the Gujarat Titans in the final.', 'route': 'summary', 'summary_result': 'The Chennai Super Kings (CSK) won the 2023 IPL by defeating the Gujarat Titans in the final.', 'final_result': 'The Chennai Super Kings (CSK) won the 2023 IPL by defeating the Gujarat Titans in the final.', 'retries': 0, 'valid': True}
exiting......


In [171]:
agent_call()


[DECOMPOSITION NODE]
[DECOMPOSED TASKS]
[ "What is 25% of 80?" ]

[ROUTER NODE]
\Route:  math

[MATH NODE]

[TOOL: Percentage Calculator]

[COMBINE NODE]
Final Result: 
25% of 80 is 20.

[VALIDATE NODE]

Final Answer: {'query': '[ "What is 25% of 80?" ]', 'route': 'math', 'math_result': '25% of 80 is 20.', 'final_result': '25% of 80 is 20.', 'retries': 0, 'valid': True}
exiting......
